# So sánh OCR tiếng Việt — CPU

Chỉ đọc hai output `FULL_BENCHMARK`. Ghép theo `(sample_id, image_sha256)` và chặn nếu revision, manifest, nhãn hoặc tập mẫu khác. Không tạo số đo giả.

In [ ]:
from pathlib import Path
import sys
LIB = Path('/kaggle/working/ocr_benchmark_lib')
LIB.mkdir(parents=True, exist_ok=True)
(LIB / 'benchmark_core.py').write_text('"""Shared, fail-closed dataset and measurement code for Kaggle OCR notebooks."""\nfrom __future__ import annotations\n\nimport base64\nimport csv\nimport gc\nimport hashlib\nimport io\nimport json\nimport os\nimport platform\nimport re\nimport shutil\nimport subprocess\nimport threading\nimport time\nimport unicodedata\nimport urllib.request\nfrom pathlib import Path\nfrom statistics import median\nfrom typing import Any, Callable\n\nDATASET_ID = "wulipc/CC-OCR"\nDATASET_FILE = "multi_lan_ocr/Vietnamese/Vietnamese_150.tsv"\nEXPECTED_COLUMNS = {"index", "image", "question", "answer"}\nMODEL_IDS = {"paddle": "PaddlePaddle/PaddleOCR-VL-1.6", "deepseek": "deepseek-ai/DeepSeek-OCR-2"}\nTERMINAL = {"completed", "failed", "oom", "timeout"}\n\n\ndef write_json(path: Path, data: Any) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str), encoding="utf-8")\n\n\ndef sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as fh:\n        for block in iter(lambda: fh.read(1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef environment() -> dict[str, Any]:\n    import psutil\n    import torch\n    result: dict[str, Any] = {\n        "python": platform.python_version(), "platform": platform.platform(),\n        "torch": torch.__version__, "torch_cuda_runtime": torch.version.cuda,\n        "cuda_available": torch.cuda.is_available(),\n        "ram_total_bytes": psutil.virtual_memory().total,\n        "ram_available_bytes": psutil.virtual_memory().available,\n        "disk_free_bytes": shutil.disk_usage("/kaggle/working").free,\n        "gpu": [], "internet": False,\n    }\n    if torch.cuda.is_available():\n        for i in range(torch.cuda.device_count()):\n            p = torch.cuda.get_device_properties(i)\n            free, total = torch.cuda.mem_get_info(i)\n            result["gpu"].append({"index": i, "name": p.name, "compute_capability": list(p.major_minor) if hasattr(p, "major_minor") else [p.major, p.minor], "total_vram_bytes": total, "free_vram_bytes": free})\n    try:\n        result["nvidia_smi"] = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,memory.used,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=10).stdout.strip()\n    except (OSError, subprocess.TimeoutExpired) as exc:\n        result["nvidia_smi_error"] = str(exc)\n    try:\n        with urllib.request.urlopen("https://huggingface.co", timeout=10) as response:\n            result["internet"] = response.status == 200\n    except Exception as exc:\n        result["internet_error"] = str(exc)\n    return result\n\n\ndef require_t4(env: dict[str, Any]) -> None:\n    if not env["cuda_available"] or not env["gpu"]:\n        raise RuntimeError("GPU CUDA không khả dụng; dừng benchmark GPU.")\n    gpu = env["gpu"][0]\n    if "T4" not in gpu["name"] or gpu["compute_capability"] != [7, 5]:\n        raise RuntimeError(f"Cần NVIDIA T4 trên cuda:0, thực tế: {gpu}")\n\n\ndef download_tsv(workdir: Path, revision: str | None = None) -> tuple[Path, str]:\n    from huggingface_hub import HfApi, hf_hub_download\n    revision = revision or HfApi().dataset_info(DATASET_ID).sha\n    path = Path(hf_hub_download(repo_id=DATASET_ID, repo_type="dataset", filename=DATASET_FILE, revision=revision, local_dir=workdir / "hf_dataset"))\n    return path, revision\n\n\ndef inspect_tsv(path: Path) -> dict[str, Any]:\n    """Read actual schema before decoding. Never print Base64 image data."""\n    csv.field_size_limit(2**31 - 1)  # Base64 image field can be megabytes.\n    with path.open("r", encoding="utf-8-sig", newline="") as fh:\n        reader = csv.DictReader(fh, delimiter="\\t")\n        fields = reader.fieldnames or []\n        if not EXPECTED_COLUMNS.issubset(fields):\n            raise ValueError(f"TSV schema mismatch: {fields}; required {sorted(EXPECTED_COLUMNS)}")\n        preview = []\n        for row in reader:\n            preview.append({"index": row.get("index"), "question": row.get("question"), "answer": row.get("answer"), "image_chars": len(row.get("image") or "")})\n            if len(preview) == 5:\n                break\n    return {"columns": fields, "preview": preview, "tsv_sha256": sha256(path)}\n\n\ndef make_manifest(path: Path, outdir: Path) -> tuple[list[dict[str, str]], list[dict[str, str]]]:\n    from PIL import Image, UnidentifiedImageError\n    csv.field_size_limit(2**31 - 1)\n    images = outdir / "images"\n    images.mkdir(parents=True, exist_ok=True)\n    rows: list[dict[str, str]] = []\n    rejected: list[dict[str, str]] = []\n    seen_ids: set[str] = set()\n    seen_hashes: dict[str, str] = {}\n    with path.open("r", encoding="utf-8-sig", newline="") as fh:\n        reader = csv.DictReader(fh, delimiter="\\t")\n        if not EXPECTED_COLUMNS.issubset(reader.fieldnames or []):\n            raise ValueError("TSV schema changed; inspect before parsing")\n        for ordinal, item in enumerate(reader):\n            sample_id = str(item.get("index", "")).strip()\n            reason = ""\n            try:\n                if not sample_id or sample_id in seen_ids:\n                    raise ValueError("empty or duplicate sample ID")\n                seen_ids.add(sample_id)\n                answer = item.get("answer")\n                if not isinstance(answer, str) or not answer.strip():\n                    raise ValueError("empty ground truth")\n                encoded = item.get("image")\n                if not isinstance(encoded, str) or not encoded.strip():\n                    raise ValueError("empty image")\n                if encoded.startswith("data:"):\n                    encoded = encoded.split(",", 1)[1]\n                payload = base64.b64decode(encoded, validate=True)\n                with Image.open(io.BytesIO(payload)) as img:\n                    img.verify()\n                with Image.open(io.BytesIO(payload)) as img:\n                    width, height = img.size\n                    fmt = (img.format or "PNG").lower()\n                    if width < 1 or height < 1:\n                        raise ValueError("invalid dimensions")\n                digest = hashlib.sha256(payload).hexdigest()\n                suffix = ".jpg" if fmt in {"jpeg", "jpg"} else f".{fmt}"\n                image_path = images / f"{ordinal:03d}{suffix}"\n                image_path.write_bytes(payload)\n                rows.append({"sample_id": sample_id, "ordinal": str(ordinal), "image_path": str(image_path), "image_sha256": digest,\n                             "width": str(width), "height": str(height), "ground_truth": answer, "question": item.get("question", ""),\n                             "duplicate_of": seen_hashes.get(digest, "")})\n                seen_hashes.setdefault(digest, sample_id)\n            except (ValueError, TypeError, KeyError, UnidentifiedImageError, OSError, base64.binascii.Error) as exc:\n                reason = str(exc)\n            if reason:\n                rejected.append({"ordinal": str(ordinal), "sample_id": sample_id, "reason": reason})\n    if not rows:\n        raise RuntimeError("No valid samples")\n    fields = list(rows[0])\n    with (outdir / "manifest.csv").open("w", encoding="utf-8", newline="") as fh:\n        writer = csv.DictWriter(fh, fieldnames=fields)\n        writer.writeheader(); writer.writerows(rows)\n    with (outdir / "rejected_samples.csv").open("w", encoding="utf-8", newline="") as fh:\n        writer = csv.DictWriter(fh, fieldnames=["ordinal", "sample_id", "reason"])\n        writer.writeheader(); writer.writerows(rejected)\n    return rows, rejected\n\n\ndef manifest_fingerprint(rows: list[dict[str, str]]) -> str:\n    canonical = [{k: row[k] for k in ("sample_id", "image_sha256", "width", "height", "ground_truth")}\n                 for row in rows]\n    payload = json.dumps(canonical, ensure_ascii=False, sort_keys=True).encode("utf-8")\n    return hashlib.sha256(payload).hexdigest()\n\n\ndef normalize(text: str, mode: str) -> str:\n    if mode == "strict":\n        return text\n    if mode != "normalized":\n        raise ValueError(mode)\n    return " ".join(unicodedata.normalize("NFC", text).replace("\\r\\n", "\\n").replace("\\r", "\\n").split())\n\n\ndef plain_text(raw: str) -> str:\n    """Conservative Markdown removal; keep visible characters and punctuation."""\n    raw = re.sub(r"!\\[[^]]*\\]\\([^)]*\\)", "", raw)\n    raw = re.sub(r"\\[([^]]+)\\]\\([^)]*\\)", r"\\1", raw)\n    raw = re.sub(r"(?m)^\\s{0,3}(?:#{1,6}\\s+|>\\s+|[-*+]\\s+|\\d+\\.\\s+)", "", raw)\n    raw = raw.replace("**", "").replace("__", "").replace("`", "")\n    return raw\n\n\ndef _distance(a: list[str] | str, b: list[str] | str) -> int:\n    try:\n        from rapidfuzz.distance import Levenshtein\n        return Levenshtein.distance(a, b)\n    except ImportError:\n        previous = list(range(len(b) + 1))\n        for i, left in enumerate(a, start=1):\n            current = [i]\n            for j, right in enumerate(b, start=1):\n                current.append(min(current[-1] + 1, previous[j] + 1, previous[j-1] + (left != right)))\n            previous = current\n        return previous[-1]\n\n\ndef score(records: list[dict[str, Any]], mode: str) -> dict[str, Any]:\n    if not records:\n        return {"status": "NOT_EVALUATED", "n": 0}\n    chars_ref = chars_edit = words_ref = words_edit = exact = empty = failed = 0\n    per_image = []\n    for row in records:\n        gt = normalize(row["ground_truth"], mode)\n        ok = row["status"] == "completed"\n        pred = normalize(row.get("prediction_text", "") if ok else "", mode)\n        failed += not ok\n        empty += not pred\n        exact += pred == gt and ok\n        cd = _distance(gt, pred)\n        wd = _distance(gt.split(), pred.split())\n        chars_ref += len(gt); chars_edit += cd\n        words_ref += len(gt.split()); words_edit += wd\n        per_image.append(cd / max(1, len(gt)))\n    n = len(records)\n    return {"status": "MEASURED", "n": n, "corpus_cer": chars_edit / max(1, chars_ref),\n            "corpus_whitespace_wer": words_edit / max(1, words_ref), "median_per_image_cer": median(per_image),\n            "exact_match_rate": exact / n, "empty_prediction_rate": empty / n, "runtime_error_rate": failed / n,\n            "failed_policy": "failed/oom/timeout => empty prediction for end-to-end quality; reported separately"}\n\n\ndef read_jsonl(path: Path) -> list[dict[str, Any]]:\n    if not path.exists():\n        return []\n    rows = []\n    with path.open(encoding="utf-8") as fh:\n        for line in fh:\n            if line.strip():\n                rows.append(json.loads(line))\n    return rows\n\n\ndef append_jsonl(path: Path, row: dict[str, Any]) -> None:\n    with path.open("a", encoding="utf-8") as fh:\n        fh.write(json.dumps(row, ensure_ascii=False, default=str) + "\\n")\n        fh.flush(); os.fsync(fh.fileno())\n\n\nclass DeviceVRAMSampler:\n    """Sample whole-device NVML usage; includes other processes and may miss short spikes."""\n    def __init__(self) -> None:\n        self.peak: int | None = None\n        self.stop = threading.Event()\n        self.thread: threading.Thread | None = None\n        try:\n            import pynvml\n            pynvml.nvmlInit()\n            self.pynvml = pynvml\n            self.handle = pynvml.nvmlDeviceGetHandleByIndex(0)\n        except Exception:\n            self.pynvml = None\n\n    def start(self) -> None:\n        if self.pynvml is None:\n            return\n        def poll() -> None:\n            while not self.stop.is_set():\n                try:\n                    used = int(self.pynvml.nvmlDeviceGetMemoryInfo(self.handle).used)\n                    self.peak = max(self.peak or 0, used)\n                except Exception:\n                    pass\n                self.stop.wait(.05)\n        self.thread = threading.Thread(target=poll, daemon=True)\n        self.thread.start()\n\n    def finish(self) -> int | None:\n        self.stop.set()\n        if self.thread:\n            self.thread.join(timeout=1)\n        return self.peak\n\n\ndef export_predictions(rows: list[dict[str, Any]], outdir: Path) -> None:\n    fields = ["sample_id", "image_sha256", "ground_truth", "status", "prediction_raw", "prediction_text", "latency_s", "peak_allocated_bytes", "peak_reserved_bytes", "peak_device_vram_bytes", "ram_rss_bytes", "truncated", "error"]\n    with (outdir / "predictions.csv").open("w", encoding="utf-8", newline="") as fh:\n        writer = csv.DictWriter(fh, fieldnames=fields, extrasaction="ignore")\n        writer.writeheader(); writer.writerows(rows)\n    with (outdir / "failures.csv").open("w", encoding="utf-8", newline="") as fh:\n        writer = csv.DictWriter(fh, fieldnames=fields, extrasaction="ignore")\n        writer.writeheader(); writer.writerows(r for r in rows if r["status"] != "completed")\n\n\ndef performance(rows: list[dict[str, Any]]) -> dict[str, Any]:\n    import numpy as np\n    good = [float(r["latency_s"]) for r in rows if r["status"] == "completed"]\n    all_times = [float(r["latency_s"]) for r in rows]\n    peaks = [int(r["peak_device_vram_bytes"]) for r in rows if r.get("peak_device_vram_bytes") is not None]\n    return {"completed_count": len(good), "total_inference_seconds": sum(all_times),\n            "throughput_images_per_second": len(good) / sum(all_times) if all_times else None,\n            "latency_p50_s": float(np.percentile(good, 50)) if good else None,\n            "latency_p90_s": float(np.percentile(good, 90)) if good else None,\n            "latency_p95_s": float(np.percentile(good, 95)) if good else None,\n            "peak_device_vram_bytes": max(peaks) if peaks else None,\n            "vram_note": "NVML whole-device sampled at 50 ms; includes other processes and can miss brief peaks"}\n\n\ndef run_benchmark(manifest: list[dict[str, str]], outdir: Path, infer: Callable[[Path], tuple[str, bool | None]],\n                  sync: Callable[[], None], gpu_peak: Callable[[], tuple[int | None, int | None, int | None]],\n                  mode: str, warmup: Path, max_seconds: float = 300.0,\n                  reset_peak: Callable[[], None] | None = None) -> list[dict[str, Any]]:\n    """Append a terminal record after each sample. Warm-up is excluded from measured rows."""\n    import psutil\n    outdir.mkdir(parents=True, exist_ok=True)\n    chosen = manifest[:5] if mode == "SMOKE_TEST" else manifest\n    path = outdir / "predictions.jsonl"\n    old = read_jsonl(path)\n    ids = [(x["sample_id"], x["image_sha256"]) for x in chosen]\n    if len(ids) != len(set(ids)):\n        raise RuntimeError("duplicate sample keys")\n    allowed = set(ids)\n    if any((r["sample_id"], r["image_sha256"]) not in allowed for r in old):\n        raise RuntimeError("checkpoint contains different dataset/mode")\n    if len({(r["sample_id"], r["image_sha256"]) for r in old}) != len(old):\n        raise RuntimeError("duplicate checkpoint rows; resolve manually")\n    if any(r.get("status") not in TERMINAL for r in old):\n        raise RuntimeError("checkpoint contains unknown status")\n    done = {(r["sample_id"], r["image_sha256"]) for r in old if r["status"] in TERMINAL}\n    if len(done) < len(chosen):\n        sync(); start = time.perf_counter()\n        infer(warmup); sync()\n        warmup_result = {"image_path": str(warmup), "seconds": time.perf_counter() - start,\n                         "resume": bool(old)}\n        write_json(outdir / "warmup.json", warmup_result)\n        append_jsonl(outdir / "warmup_history.jsonl", warmup_result)\n    for item in chosen:\n        key = (item["sample_id"], item["image_sha256"])\n        if key in done:\n            continue\n        sync()\n        if reset_peak:\n            reset_peak()\n        sampler = DeviceVRAMSampler()\n        sampler.start()\n        start = time.perf_counter()\n        record: dict[str, Any] = {"sample_id": key[0], "image_sha256": key[1], "ground_truth": item["ground_truth"], "status": "failed", "prediction_raw": "", "prediction_text": "", "truncated": False, "error": ""}\n        try:\n            raw, truncated = infer(Path(item["image_path"]))\n            if not isinstance(raw, str):\n                raise TypeError(f"inference returned {type(raw).__name__}; expected str")\n            sync()\n            record.update(status="completed", prediction_raw=raw, prediction_text=plain_text(raw), truncated=truncated)\n        except Exception as exc:\n            sync()\n            record.update(status="oom" if "out of memory" in str(exc).lower() else "failed", error=f"{type(exc).__name__}: {exc}"[:2000])\n            gc.collect()\n            try:\n                import torch\n                torch.cuda.empty_cache()\n            except ImportError:\n                pass\n        elapsed = time.perf_counter() - start\n        device_peak = sampler.finish()\n        if elapsed > max_seconds and record["status"] == "completed":\n            record["status"] = "timeout"  # post hoc threshold; no hard preemption\n        a, r, d = gpu_peak()\n        record.update(latency_s=elapsed, peak_allocated_bytes=a, peak_reserved_bytes=r, peak_device_vram_bytes=device_peak or d,\n                      ram_rss_bytes=psutil.Process().memory_info().rss)\n        append_jsonl(path, record)\n        done.add(key)\n        if record["status"] == "oom":\n            break  # repeated OOM is unsafe; preserve checkpoint\n    rows = read_jsonl(path)\n    export_predictions(rows, outdir)\n    write_json(outdir / "metrics.json", {"run_status": "MEASURED" if len(rows) == len(chosen) else "PARTIAL",\n                                         "expected_samples": len(chosen), "recorded_samples": len(rows),\n                                         **{m: score(rows, m) for m in ("strict", "normalized")},\n                                         "performance": performance(rows)})\n    return rows\n', encoding='utf-8')
(LIB / 'compare.py').write_text('"""CPU-only comparison. Refuses to produce a full result from partial or mismatched runs."""\nfrom __future__ import annotations\n\nimport csv\nimport difflib\nimport html\nimport json\nimport re\nimport statistics\nimport unicodedata\nfrom pathlib import Path\nfrom typing import Any\n\nfrom benchmark_core import _distance, normalize, read_jsonl, score, write_json\n\n\ndef _read_run(folder: Path) -> tuple[dict[str, Any], list[dict[str, Any]]]:\n    config = json.loads((folder / "run_manifest.json").read_text(encoding="utf-8"))\n    return config, read_jsonl(folder / "predictions.jsonl")\n\n\ndef _key(row: dict[str, Any]) -> tuple[str, str]:\n    return str(row["sample_id"]), str(row["image_sha256"])\n\n\ndef verify_pair(pa: Path, pb: Path) -> tuple[list[dict[str, Any]], list[dict[str, Any]], dict[str, Any]]:\n    a_meta, a = _read_run(pa)\n    b_meta, b = _read_run(pb)\n    if a_meta["model_id"] != "PaddlePaddle/PaddleOCR-VL-1.6" or b_meta["model_id"] != "deepseek-ai/DeepSeek-OCR-2":\n        raise ValueError("Wrong model IDs")\n    for field in ("dataset_revision", "dataset_sha256", "manifest_sha256", "mode"):\n        if a_meta[field] != b_meta[field]:\n            raise ValueError(f"Mismatch in {field}")\n    if a_meta["mode"] != "FULL_BENCHMARK":\n        raise ValueError("Comparison requires FULL_BENCHMARK outputs")\n    aa, bb = {_key(x): x for x in a}, {_key(x): x for x in b}\n    if len(aa) != len(a) or len(bb) != len(b):\n        raise ValueError("Duplicate checkpoint row")\n    if aa.keys() != bb.keys():\n        missing_a = sorted(bb.keys() - aa.keys())\n        missing_b = sorted(aa.keys() - bb.keys())\n        raise ValueError(f"Incomplete common set: missing Paddle={missing_a[:10]}, DeepSeek={missing_b[:10]}")\n    expected = a_meta["valid_sample_count"]\n    if len(aa) != expected:\n        raise ValueError(f"Incomplete benchmark: {len(aa)} / {expected}")\n    with (pa / "manifest.csv").open(encoding="utf-8", newline="") as fh:\n        manifest = {_key(r): r for r in csv.DictReader(fh)}\n    if len(manifest) != expected or aa.keys() != manifest.keys():\n        raise ValueError("Prediction keys do not match full manifest")\n    for key in aa:\n        if aa[key]["ground_truth"] != bb[key]["ground_truth"]:\n            raise ValueError(f"Ground truth mismatch for {key}")\n        if aa[key]["ground_truth"] != manifest[key]["ground_truth"]:\n            raise ValueError(f"Ground truth differs from manifest for {key}")\n        if aa[key]["status"] not in {"completed", "failed", "oom", "timeout"} or bb[key]["status"] not in {"completed", "failed", "oom", "timeout"}:\n            raise ValueError(f"Unknown status for {key}")\n    for folder in (pa, pb):\n        env = json.loads((folder / "environment.json").read_text(encoding="utf-8"))\n        gpu = env.get("gpu", [{}])[0]\n        if "T4" not in gpu.get("name", "") or gpu.get("compute_capability") != [7, 5]:\n            raise ValueError(f"Non-T4 or unverified GPU environment: {folder}")\n    keys = sorted(aa)\n    return [aa[k] for k in keys], [bb[k] for k in keys], a_meta\n\n\ndef category_flags(gt: str, pred: str) -> dict[str, bool]:\n    """Heuristic tags from actual edits; not mutually exclusive or causal labels."""\n    g, p = normalize(gt, "normalized"), normalize(pred, "normalized")\n    strip_marks = lambda s: "".join(c for c in unicodedata.normalize("NFD", s) if not unicodedata.combining(c))\n    return {\n        "diacritic_only": g != p and strip_marks(g).casefold() == strip_marks(p).casefold(),\n        "d_vs_stroke": g != p and g.replace("đ", "d").replace("Đ", "D") == p.replace("đ", "d").replace("Đ", "D"),\n        "case_only": g != p and g.casefold() == p.casefold(),\n        "digit_difference": re.sub(r"\\D", "", g) != re.sub(r"\\D", "", p),\n        "punctuation_difference": "".join(c for c in g if unicodedata.category(c).startswith("P")) != "".join(c for c in p if unicodedata.category(c).startswith("P")),\n        "possible_omission": len(p) < len(g) and len(p) < 0.8 * len(g),\n        "possible_addition_or_repetition": len(p) > 1.2 * len(g),\n    }\n\n\ndef _cer(gt: str, pred: str) -> float:\n    return _distance(gt, pred) / max(1, len(gt))\n\n\ndef _metric_value(rows: list[dict[str, Any]], name: str) -> float | None:\n    import numpy as np\n    good = [r for r in rows if r["status"] == "completed" and r.get("latency_s") is not None]\n    if name == "throughput":\n        total = sum(float(r["latency_s"]) for r in rows if r.get("latency_s") is not None)\n        return len(good) / total if total else None\n    if name.startswith("latency"):\n        q = 50 if name == "latency_p50" else 95\n        return float(np.percentile([float(r["latency_s"]) for r in good], q)) if good else None\n    if name == "peak_vram":\n        vals = [r.get("peak_device_vram_bytes") or r.get("peak_reserved_bytes") for r in rows]\n        return max((int(v) for v in vals if v is not None), default=None)\n    return None\n\n\ndef _plot(a: list[dict[str, Any]], b: list[dict[str, Any]], out: Path, sa: dict[str, Any], sb: dict[str, Any]) -> None:\n    import matplotlib.pyplot as plt\n    import numpy as np\n    plt.style.use("default")\n    names = ["PaddleOCR-VL 1.6", "DeepSeek-OCR 2"]\n    fig, ax = plt.subplots(figsize=(7, 4))\n    x = np.arange(3)\n    ax.bar(x - .18, [sa["corpus_cer"], sa["corpus_whitespace_wer"], sa["exact_match_rate"]], .36, label=names[0])\n    ax.bar(x + .18, [sb["corpus_cer"], sb["corpus_whitespace_wer"], sb["exact_match_rate"]], .36, label=names[1])\n    ax.set_xticks(x, ["CER", "WER", "Exact match"]); ax.legend(); ax.set_ylabel("Rate")\n    fig.tight_layout(); fig.savefig(out / "quality.png", dpi=180); plt.close(fig)\n    for title, values, filename in [\n        ("Per-image CER", [[_cer(normalize(r["ground_truth"], "normalized"), normalize(r.get("prediction_text", "") if r["status"] == "completed" else "", "normalized")) for r in rows] for rows in (a,b)], "cer_boxplot.png"),\n        ("Latency", [[float(r["latency_s"]) for r in rows if r["status"] == "completed"] for rows in (a,b)], "latency_histogram.png")]:\n        fig, ax = plt.subplots(figsize=(7, 4))\n        if filename.startswith("cer"):\n            ax.boxplot(values, tick_labels=names, showfliers=True)\n        else:\n            for v, label in zip(values, names):\n                if v: ax.hist(v, bins=min(20, max(3, len(v)//3)), alpha=.55, label=label)\n            ax.legend(); ax.set_xlabel("Seconds")\n        ax.set_title(title); fig.tight_layout(); fig.savefig(out / filename, dpi=180); plt.close(fig)\n    fig, ax = plt.subplots(figsize=(7, 4))\n    for rows, label in zip((a,b), names):\n        xs = [float(r["latency_s"]) for r in rows if r["status"] == "completed"]\n        ys = [_cer(normalize(r["ground_truth"], "normalized"), normalize(r.get("prediction_text", ""), "normalized")) for r in rows if r["status"] == "completed"]\n        ax.scatter(xs, ys, label=label, alpha=.65)\n    ax.set(xlabel="Latency (s)", ylabel="CER"); ax.legend(); fig.tight_layout(); fig.savefig(out / "cer_vs_latency.png", dpi=180); plt.close(fig)\n    fig, ax = plt.subplots(figsize=(7, 4))\n    vram_values = [_metric_value(rows, "peak_vram") for rows in (a,b)]\n    ax.bar(names, [(v or 0) / 2**30 for v in vram_values])\n    for i, v in enumerate(vram_values):\n        if v is None: ax.text(i, 0, "NOT_EVALUATED", ha="center", va="bottom")\n    ax.set_ylabel("Peak reported VRAM (GiB)"); fig.tight_layout(); fig.savefig(out / "vram.png", dpi=180); plt.close(fig)\n    wins = [0,0,0]\n    for ra, rb in zip(a,b):\n        ca = _cer(normalize(ra["ground_truth"], "normalized"), normalize(ra.get("prediction_text", "") if ra["status"] == "completed" else "", "normalized"))\n        cb = _cer(normalize(rb["ground_truth"], "normalized"), normalize(rb.get("prediction_text", "") if rb["status"] == "completed" else "", "normalized"))\n        wins[0 if ca < cb else 1 if cb < ca else 2] += 1\n    fig, ax = plt.subplots(figsize=(7, 4)); ax.bar(["Paddle wins", "DeepSeek wins", "Tie"], wins)\n    ax.set_ylabel("Images"); fig.tight_layout(); fig.savefig(out / "wins.png", dpi=180); plt.close(fig)\n\n\ndef compare(paddle_dir: Path, deepseek_dir: Path, out: Path) -> dict[str, Any]:\n    out.mkdir(parents=True, exist_ok=True)\n    a, b, meta = verify_pair(paddle_dir, deepseek_dir)\n    deepseek_meta = json.loads((deepseek_dir / "run_manifest.json").read_text(encoding="utf-8"))\n    env_a = json.loads((paddle_dir / "environment.json").read_text(encoding="utf-8"))\n    env_b = json.loads((deepseek_dir / "environment.json").read_text(encoding="utf-8"))\n    strict_a, strict_b = score(a, "strict"), score(b, "strict")\n    sa, sb = score(a, "normalized"), score(b, "normalized")\n    metrics = [\n        ("CER", sa["corpus_cer"], sb["corpus_cer"], False),\n        ("WER (whitespace)", sa["corpus_whitespace_wer"], sb["corpus_whitespace_wer"], False),\n        ("Exact Match", sa["exact_match_rate"], sb["exact_match_rate"], True),\n        ("Latency p50", _metric_value(a,"latency_p50"), _metric_value(b,"latency_p50"), False),\n        ("Latency p95", _metric_value(a,"latency_p95"), _metric_value(b,"latency_p95"), False),\n        ("Throughput", _metric_value(a,"throughput"), _metric_value(b,"throughput"), True),\n        ("Peak VRAM", _metric_value(a,"peak_vram"), _metric_value(b,"peak_vram"), False),\n    ]\n    table = []\n    for name, x, y, higher in metrics:\n        better = "NOT_EVALUATED" if x is None or y is None else "Tie" if x == y else ("PaddleOCR-VL 1.6" if (x > y) == higher else "DeepSeek-OCR 2")\n        table.append({"Metric": name, "PaddleOCR-VL 1.6": x, "DeepSeek-OCR 2": y, "Better": better})\n    summary = {"status": "MEASURED", "dataset_revision": meta["dataset_revision"], "n": len(a),\n               "strict": {"paddle": strict_a, "deepseek": strict_b}, "normalized": {"paddle": sa, "deepseek": sb}, "table": table}\n    combined = []\n    for ra, rb in zip(a,b):\n        gt = ra["ground_truth"]\n        ca = _cer(normalize(gt,"normalized"), normalize(ra.get("prediction_text","") if ra["status"] == "completed" else "","normalized"))\n        cb = _cer(normalize(gt,"normalized"), normalize(rb.get("prediction_text","") if rb["status"] == "completed" else "","normalized"))\n        combined.append({"sample_id": ra["sample_id"], "image_sha256": ra["image_sha256"], "ground_truth": gt,\n                         "paddle_prediction": ra.get("prediction_text", ""), "deepseek_prediction": rb.get("prediction_text", ""),\n                         "paddle_status": ra["status"], "deepseek_status": rb["status"], "paddle_cer": ca, "deepseek_cer": cb,\n                         "paddle_flags": json.dumps(category_flags(gt,ra.get("prediction_text","")),ensure_ascii=False),\n                         "deepseek_flags": json.dumps(category_flags(gt,rb.get("prediction_text","")),ensure_ascii=False)})\n    with (out / "comparison.csv").open("w", encoding="utf-8", newline="") as fh:\n        writer = csv.DictWriter(fh, fieldnames=list(combined[0])); writer.writeheader(); writer.writerows(combined)\n    _plot(a,b,out,sa,sb)\n    categories = {model: {kind: sum(category_flags(r["ground_truth"], r.get("prediction_text", "") if r["status"] == "completed" else "")[kind] for r in rows)\n                          for kind in category_flags("", "")}\n                  for model, rows in (("PaddleOCR-VL 1.6", a), ("DeepSeek-OCR 2", b))}\n    summary["error_categories_heuristic"] = categories\n    write_json(out / "comparison_summary.json", summary)\n    report = ["# Báo cáo OCR tiếng Việt", "",\n              "## 1. Dataset và giới hạn nghiên cứu", "",\n              f"CC-OCR Vietnamese_150.tsv, commit `{meta[\'dataset_revision\']}`, SHA256 TSV `{meta[\'dataset_sha256\']}`, {len(a)} ảnh hợp lệ. Chỉ tập này được chấm; số mẫu hợp lệ có thể khác 150. Nhãn `answer` không được kiểm định độc lập ngoài cổng xác nhận trong notebook.", "",\n              "## 2. Mô hình", "",\n              f"PaddleOCR-VL 1.6: commit tham chiếu `{meta[\'model_revision\']}`, pipeline chính thức gồm layout + VLM. DeepSeek-OCR 2: commit `{deepseek_meta[\'model_revision\']}`, prompt Free OCR, FP16/eager trên T4. Bản vá: `{deepseek_meta.get(\'patch\')}`.", "",\n              "## 3. Phần cứng và môi trường", "",\n              f"Paddle: {env_a[\'gpu\'][0][\'name\']}, Python {env_a.get(\'python\')}, Torch {env_a.get(\'torch\')}, CUDA {env_a.get(\'torch_cuda_runtime\')}, driver {env_a.get(\'nvidia_smi\')}. DeepSeek: {env_b[\'gpu\'][0][\'name\']}, Python {env_b.get(\'python\')}, Torch {env_b.get(\'torch\')}, CUDA {env_b.get(\'torch_cuda_runtime\')}, driver {env_b.get(\'nvidia_smi\')}.", "",\n              "## 4. Phương pháp thực nghiệm", "",\n              "Ảnh, nhãn và sample ID/hash được kiểm tra chung. Batch 1, cuda:0, seed 42, warm-up riêng. Strict giữ nguyên ký tự; normalized chỉ NFC và khoảng trắng. CER/WER dùng edit distance; WER token theo khoảng trắng. Lỗi runtime/OOM/timeout được tính như dự đoán rỗng trong chất lượng đầu cuối và báo riêng.", "",\n              "## 5. Bảng kết quả normalized", "",\n              "| Metric | PaddleOCR-VL 1.6 | DeepSeek-OCR 2 | Better |", "|---|---:|---:|---|"]\n    for row in table:\n        fmt = lambda x: "NOT_EVALUATED" if x is None else f"{x:.4f}"\n        report.append(f"| {row[\'Metric\']} | {fmt(row[\'PaddleOCR-VL 1.6\'])} | {fmt(row[\'DeepSeek-OCR 2\'])} | {row[\'Better\']} |")\n    report += ["", f"Strict CER: Paddle {strict_a[\'corpus_cer\']:.4f}, DeepSeek {strict_b[\'corpus_cer\']:.4f}; strict whitespace WER: Paddle {strict_a[\'corpus_whitespace_wer\']:.4f}, DeepSeek {strict_b[\'corpus_whitespace_wer\']:.4f}.",\n               "", "## 6. Phân tích lỗi tiếng Việt", "",\n               "Các đếm dưới đây là heuristic từ chuỗi khác biệt, có thể chồng lấn và không chứng minh nguyên nhân lỗi."]\n    for model, counts in categories.items():\n        report.append(f"- {model}: " + ", ".join(f"{k}={v}" for k, v in counts.items()))\n    report += ["", "Xem `error_analysis.html` để đối chiếu ảnh, nhãn, hai dự đoán và diff của tối đa 10 ảnh lỗi theo quy tắc chọn cố định.",\n               "", "## 7. Độ chính xác và chi phí tài nguyên", "",\n               f"Runtime error rate: Paddle {sa[\'runtime_error_rate\']:.4f}, DeepSeek {sb[\'runtime_error_rate\']:.4f}. Latency và VRAM nằm trong bảng; NVML đo toàn GPU mỗi 50 ms nên đây là ước lượng có giới hạn. Throughput là số ảnh thành công chia tổng thời gian inference của mọi ảnh đã thử.",\n               "", "## 8. Threats to validity", "",\n               "Tập chỉ có một ngôn ngữ và tối đa 150 ảnh. Nhãn chưa có kiểm định độc lập. FP16 DeepSeek là bản port chưa được tác giả xác nhận. Paddle gồm layout analysis; so sánh này không tách vision encoder. Markdown-to-text có thể ảnh hưởng CER. Hai phiên Kaggle có thể khác driver/tải GPU. NVML có thể bỏ lỡ peak ngắn và gồm tiến trình khác. Pipeline Paddle có thể tải artifact qua registry khác với HF commit tham chiếu.",\n               "", "## 9. Kết luận", "",\n               f"Trên tập đã đo: CER tốt hơn: {table[0][\'Better\']}; WER tốt hơn: {table[1][\'Better\']}; p50 nhanh hơn: {table[3][\'Better\']}; peak VRAM thấp hơn: {table[6][\'Better\']}. Đây là kết luận theo từng tiêu chí, không tự suy ra một lựa chọn triển khai duy nhất. Khi triển khai thực tế, cần thêm dữ liệu đích, độ ổn định và chi phí vận hành."]\n    (out / "comparison_report.md").write_text("\\n".join(report), encoding="utf-8")\n    # Deterministic top errors: largest best-model CER, then ID.\n    examples = sorted((r for r in combined if max(r["paddle_cer"], r["deepseek_cer"]) > 0),\n                      key=lambda r: (-min(r["paddle_cer"], r["deepseek_cer"]), r["sample_id"]))[:10]\n    html_rows = ["<html><meta charset=\'utf-8\'><style>body{font:16px sans-serif;max-width:1100px;margin:auto;background:#fff;color:#222}section{border-bottom:1px solid #ddd;padding:18px}img{max-width:420px}del{background:#fdd}ins{background:#dfd}pre{white-space:pre-wrap}</style><h1>Phân tích lỗi OCR</h1>"]\n    manifest_path = paddle_dir / "manifest.csv"\n    paths = {}\n    if manifest_path.exists():\n        with manifest_path.open(encoding="utf-8", newline="") as fh:\n            paths = {r["sample_id"]: Path(r["image_path"]) for r in csv.DictReader(fh)}\n    for r in examples:\n        pic = paths.get(r["sample_id"])\n        if pic and not pic.is_file():\n            pic = paddle_dir / "images" / pic.name\n        img = ""\n        if pic and pic.is_file():\n            import base64\n            mime = "image/jpeg" if pic.suffix.lower() in {".jpg",".jpeg"} else "image/png"\n            img = f"<img src=\'data:{mime};base64,{base64.b64encode(pic.read_bytes()).decode()}\'>"\n        diff = lambda pred: "".join(("<del>" if x.startswith("-") else "<ins>" if x.startswith("+") else "<span>") + html.escape(x[2:]) + ("</del>" if x.startswith("-") else "</ins>" if x.startswith("+") else "</span>") for x in difflib.ndiff(r["ground_truth"], pred))\n        flags = category_flags(r["ground_truth"], r["paddle_prediction"])\n        note = ", ".join(k for k,v in flags.items() if v) or "Lỗi ký tự khác"\n        html_rows.append(f"<section><h2>{html.escape(r[\'sample_id\'])}</h2>{img}<p>GT: {html.escape(r[\'ground_truth\'])}</p><p>Paddle CER {r[\'paddle_cer\']:.3f}: {html.escape(r[\'paddle_prediction\'])}</p><p>DeepSeek CER {r[\'deepseek_cer\']:.3f}: {html.escape(r[\'deepseek_prediction\'])}</p><p>Diff Paddle: {diff(r[\'paddle_prediction\'])}</p><p>Diff DeepSeek: {diff(r[\'deepseek_prediction\'])}</p><p>Nhận xét tự động: {html.escape(note)}</p></section>")\n    html_rows.append("</html>")\n    (out / "error_analysis.html").write_text("\\n".join(html_rows), encoding="utf-8")\n    return summary\n', encoding='utf-8')
sys.path.insert(0, str(LIB))
print('Embedded benchmark source ready:', sorted(p.name for p in LIB.iterdir()))

## 1. Cài thư viện phân tích và tìm Kaggle Inputs

Thêm hai Notebook Outputs hoặc Kaggle Datasets làm inputs. Notebook tự tìm đúng `run_manifest.json` theo model ID. Nếu nhiều bản, đặt đường dẫn tường minh bên dưới.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m','pip','install','rapidfuzz==3.14.1','numpy==2.0.2','matplotlib==3.10.3'])
from pathlib import Path
import json
ROOT = Path('/kaggle/working/vietnamese_ocr_compare')
ROOT.mkdir(parents=True, exist_ok=True)
PADDLE_OUTPUT_DIR = None  # e.g. Path('/kaggle/input/<notebook-output>/.../full_benchmark')
DEEPSEEK_OUTPUT_DIR = None
found = []
for p in Path('/kaggle/input').rglob('run_manifest.json'):
    try:
        meta = json.loads(p.read_text(encoding='utf-8'))
        found.append((p.parent, meta.get('model_id'), meta.get('mode')))
    except (OSError, ValueError): pass
print('Found:', found)
def choose(model_id):
    dirs = [d for d,m,mode in found if m == model_id and mode == 'FULL_BENCHMARK']
    if len(dirs) != 1: raise RuntimeError(f'Expected exactly one full output for {model_id}; found {dirs}')
    return dirs[0]
PADDLE_OUTPUT_DIR = Path(PADDLE_OUTPUT_DIR) if PADDLE_OUTPUT_DIR else choose('PaddlePaddle/PaddleOCR-VL-1.6')
DEEPSEEK_OUTPUT_DIR = Path(DEEPSEEK_OUTPUT_DIR) if DEEPSEEK_OUTPUT_DIR else choose('deepseek-ai/DeepSeek-OCR-2')

## 2. Kiểm tra tính công bằng và tính metrics

Strict giữ nguyên ký tự. Normalized chỉ NFC và khoảng trắng. CER là edit distance ở mức ký tự, không cắt tại 1. WER là token theo khoảng trắng, không phải tách từ tiếng Việt ngôn ngữ học. Runtime error/OOM/timeout được báo riêng và tính dự đoán rỗng cho chất lượng đầu cuối.

In [ ]:
from compare import compare
from benchmark_core import write_json
try:
    SUMMARY = compare(PADDLE_OUTPUT_DIR, DEEPSEEK_OUTPUT_DIR, ROOT)
except Exception as exc:
    write_json(ROOT / 'comparison_summary.json', {'status': 'NOT_EVALUATED', 'reason': str(exc)})
    (ROOT / 'comparison_report.md').write_text('# Chưa thể so sánh\n\nNOT_EVALUATED: ' + str(exc), encoding='utf-8')
    raise
from IPython.display import display
import pandas as pd
display(pd.DataFrame(SUMMARY['table']))

## 3. Biểu đồ và mười ca lỗi

Các ảnh lỗi chọn bằng quy tắc cố định: CER của mô hình tốt hơn trên từng ảnh cao nhất, sau đó theo sample ID. Nhãn lỗi là heuristic dựa trên edit; phải đọc diff để kiểm chứng. HTML tự chứa ảnh nếu Notebook Output có `manifest.csv` và ảnh.

In [ ]:
from IPython.display import Image, display, HTML
for name in ['quality.png','cer_boxplot.png','latency_histogram.png','cer_vs_latency.png','vram.png','wins.png']:
    display(Image(filename=str(ROOT / name)))
print('Report:', ROOT / 'comparison_report.md')
print('Error analysis:', ROOT / 'error_analysis.html')
display(HTML((ROOT / 'error_analysis.html').read_text(encoding='utf-8')))